# RAG + Knowledge Graph for Diabetes — Nguyen Ba Duy 076
Evidence-linked educational prototype. It does not provide diagnosis or treatment.

In [ ]:
%pip -q install pandas numpy scikit-learn networkx pypdf
import re, unicodedata, numpy as np, pandas as pd, networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
np.random.seed(76)


In [ ]:
SOURCES = {
    'NIDDK_OVERVIEW': 'https://www.niddk.nih.gov/health-information/diabetes/overview',
    'NIDDK_SYMPTOMS': 'https://www.niddk.nih.gov/health-information/diabetes/overview/symptoms-causes',
    'NIDDK_PREVENTION': 'https://www.niddk.nih.gov/health-information/diabetes/overview/preventing-type-2-diabetes',
    'CDC_TYPE2': 'https://www.cdc.gov/diabetes/about/about-type-2-diabetes.html',
    'CDC_RISK': 'https://www.cdc.gov/diabetes/risk-factors/index.html',
    'WHO_FACTS': 'https://www.who.int/news-room/fact-sheets/detail/diabetes',
}
KB = [
 {'id':'D1','source':SOURCES['NIDDK_OVERVIEW'],'text':'Diabetes is a condition in which blood glucose is too high. Main types include type 1, type 2 and gestational diabetes.'},
 {'id':'D2','source':SOURCES['NIDDK_SYMPTOMS'],'text':'Possible symptoms include thirst, frequent urination, fatigue, unexplained weight loss and blurred vision.'},
 {'id':'D3','source':SOURCES['CDC_RISK'],'text':'Family history, overweight, low activity and older age are associated with increased type 2 diabetes risk.'},
 {'id':'D4','source':SOURCES['NIDDK_PREVENTION'],'text':'Physical activity, weight management and healthy eating can help prevent or delay type 2 diabetes.'},
 {'id':'D5','source':SOURCES['WHO_FACTS'],'text':'Diabetes may affect the heart, blood vessels, eyes, kidneys and feet.'},
 {'id':'D6','source':SOURCES['CDC_TYPE2'],'text':'Type 2 diabetes may develop slowly and may not have noticeable symptoms.'},
]
TRIPLES = [('Diabetes', 'HAS_TYPE', 'Type 1', 'D1'), ('Diabetes', 'HAS_TYPE', 'Type 2', 'D1'), ('Diabetes', 'HAS_TYPE', 'Gestational diabetes', 'D1'), ('Diabetes', 'CHARACTERIZED_BY', 'High blood glucose', 'D1'), ('Diabetes', 'HAS_SYMPTOM', 'Thirst', 'D2'), ('Diabetes', 'HAS_SYMPTOM', 'Frequent urination', 'D2'), ('Diabetes', 'HAS_SYMPTOM', 'Fatigue', 'D2'), ('Diabetes', 'HAS_SYMPTOM', 'Unexplained weight loss', 'D2'), ('Diabetes', 'HAS_SYMPTOM', 'Blurred vision', 'D2'), ('Type 2', 'HAS_RISK_FACTOR', 'Family history', 'D3'), ('Type 2', 'HAS_RISK_FACTOR', 'Overweight', 'D3'), ('Type 2', 'HAS_RISK_FACTOR', 'Low physical activity', 'D3'), ('Type 2', 'HAS_RISK_FACTOR', 'Older age', 'D3'), ('Type 2', 'HAS_RISK_FACTOR', 'Prediabetes', 'D6'), ('Type 2', 'CAN_BE_DELAYED_BY', 'Physical activity', 'D4'), ('Type 2', 'CAN_BE_DELAYED_BY', 'Weight management', 'D4'), ('Type 2', 'CAN_BE_DELAYED_BY', 'Healthy eating', 'D4'), ('Diabetes', 'MAY_AFFECT', 'Heart', 'D5'), ('Diabetes', 'MAY_AFFECT', 'Kidneys', 'D5'), ('Diabetes', 'MAY_AFFECT', 'Eyes', 'D5'), ('Diabetes', 'MAY_AFFECT', 'Feet', 'D5'), ('Diabetes', 'MAY_AFFECT', 'Blood vessels', 'D5'), ('High blood glucose', 'MANAGED_BY', 'Glucose management', 'D5'), ('Glucose management', 'HELPS_REDUCE_RISK_FOR', 'Heart', 'D5'), ('Glucose management', 'HELPS_REDUCE_RISK_FOR', 'Kidneys', 'D5'), ('Glucose management', 'HELPS_REDUCE_RISK_FOR', 'Eyes', 'D5'), ('Physical activity', 'SUPPORTS', 'Weight management', 'D4'), ('Healthy eating', 'SUPPORTS', 'Weight management', 'D4'), ('Type 2', 'MAY_HAVE', 'No clear symptoms', 'D6'), ('Prediabetes', 'INCREASES_RISK_OF', 'Type 2', 'D6'), ('Family history', 'INCREASES_RISK_OF', 'Type 2', 'D6')]
print('documents=', len(KB), 'triples=', len(TRIPLES))


In [ ]:
# Optional PDF ingestion: upload PDFs in Colab, then extract one record per page.
from pathlib import Path
from pypdf import PdfReader
def records_from_pdfs(folder='pdf_sources'):
    records=[]
    for path in Path(folder).glob('*.pdf'):
        for page_no, page in enumerate(PdfReader(str(path)).pages, 1):
            text=(page.extract_text() or '').strip()
            if text: records.append({'id':f'{path.stem}-p{page_no}','source':str(path),'text':text})
    return records


In [ ]:
G=nx.MultiDiGraph()
for s,r,o,e in TRIPLES: G.add_edge(s,o,relation=r,evidence=e)
print('nodes=',G.number_of_nodes(),'edges=',G.number_of_edges())
def one_hop(entity): return [(entity,d['relation'],n,d['evidence']) for _,n,d in G.out_edges(entity,data=True)]
def two_hop(entity): return [fact for n in G.successors(entity) for fact in one_hop(n)]
print(one_hop('Type 2')[:5]); print(two_hop('Diabetes')[:5])


In [ ]:
vectorizer=TfidfVectorizer(ngram_range=(1,2),strip_accents='unicode')
matrix=vectorizer.fit_transform([d['text'] for d in KB])
def retrieve(question,k=3,min_score=.10):
    scores=cosine_similarity(vectorizer.transform([question]),matrix).ravel()
    idx=np.argsort(scores)[::-1][:k]
    return [(KB[i],float(scores[i])) for i in idx if scores[i]>=min_score]
def answer(question):
    hits=retrieve(question)
    if not hits: return {'answer':'The knowledge base does not contain enough evidence.','sources':[]}
    return {'answer':' '.join(f"{d['text']} [{d['id']}]" for d,_ in hits), 'sources':[d['source'] for d,_ in hits]}
answer('What are common symptoms of diabetes?')


In [ ]:
tests=[('What are symptoms of diabetes?','D2'),('What increases type 2 risk?','D3'),('How can type 2 be delayed?','D4'),('What are diabetes types?','D1'),('What drug dose should I take?','NONE')]
rows=[]
for q,gold in tests:
    ids=[d['id'] for d,_ in retrieve(q)]
    hit=(gold in ids) if gold!='NONE' else not ids
    rows.append({'question':q,'gold':gold,'retrieved':ids,'hit_at_3':int(hit)})
evaluation=pd.DataFrame(rows); display(evaluation); print('Hit@3 including abstention=',evaluation.hit_at_3.mean())


## Executed PDF knowledge base with dense local embeddings
Upload the five course PDFs listed in `knowledge_graph/pdf_kb_run.json`. The PDF files are not committed to GitHub. The executed local run processed 214 pages into 261 chunks.

In [ ]:
%pip -q install pypdf scikit-learn transformers sentencepiece accelerate
from google.colab import files
uploaded = files.upload()  # select the five course PDFs
from pathlib import Path
pdf_paths=[Path(name) for name in uploaded if name.lower().endswith(".pdf")]
print("PDF files:", [p.name for p in pdf_paths])


In [ ]:
from pypdf import PdfReader
import re, numpy as np
def clean(text): return re.sub(r"\s+", " ", text).strip()
def make_chunks(text,size=850,overlap=120):
    chunks=[]; start=0
    while start < len(text):
        end=min(len(text),start+size); value=text[start:end].strip()
        if len(value)>=80: chunks.append(value)
        if end>=len(text): break
        start=max(start+1,end-overlap)
    return chunks
pdf_records=[]
for pdf in pdf_paths:
    reader=PdfReader(str(pdf))
    for page_no,page in enumerate(reader.pages,1):
        for chunk_no,text in enumerate(make_chunks(clean(page.extract_text() or "")),1):
            pdf_records.append({"id":f"{pdf.stem}-p{page_no}-c{chunk_no}","file":pdf.name,"page":page_no,"text":text})
print("pages=",sum(len(PdfReader(str(p)).pages) for p in pdf_paths),"chunks=",len(pdf_records))


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
pdf_vectorizer=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=12000,stop_words="english")
X_tfidf=pdf_vectorizer.fit_transform([r["text"] for r in pdf_records])
dims=min(128,X_tfidf.shape[0]-1,X_tfidf.shape[1]-1)
pdf_svd=TruncatedSVD(n_components=dims,random_state=76)
pdf_norm=Normalizer(copy=False)
pdf_embeddings=pdf_norm.fit_transform(pdf_svd.fit_transform(X_tfidf))
def retrieve_pdf(question,k=4):
    q=pdf_norm.transform(pdf_svd.transform(pdf_vectorizer.transform([question])))[0]
    scores=pdf_embeddings@q; top=np.argsort(scores)[::-1][:k]
    return [{**pdf_records[i],"score":float(scores[i])} for i in top]
display(pd.DataFrame(retrieve_pdf("What is retrieval augmented generation?"))[["file","page","score","text"]])


In [ ]:
# A small instruction-tuned language model generates only from retrieved context.
from transformers import pipeline
generator=pipeline("text2text-generation",model="google/flan-t5-small",max_new_tokens=160)
def grounded_pdf_chat(question,min_score=.12):
    hits=retrieve_pdf(question,k=4)
    if not hits or hits[0]["score"]<min_score:
        return {"answer":"The knowledge base does not contain enough evidence.","sources":[]}
    context="\n".join(f"[{h['file']} p.{h['page']}] {h['text']}" for h in hits)
    prompt=f"Answer only from the context. If evidence is insufficient, say so. Cite source and page.\nQuestion: {question}\nContext:\n{context}"
    answer=generator(prompt,do_sample=False)[0]["generated_text"]
    return {"answer":answer,"sources":[{"file":h["file"],"page":h["page"],"score":h["score"]} for h in hits]}
grounded_pdf_chat("Which agents are used in the diabetes architecture?")
